# Document Processing

**Module:** 04 — RAG

Garbage in, garbage retrieved. Master ingestion, cleaning, chunking, and metadata.


## How to Use This Notebook

Read each section as a mini-lesson, run every code cell, then change inputs to stress-test your intuition. API examples use placeholders such as `YOUR_API_KEY` or `os.getenv(...)` — never hard-code secrets.

Each major topic includes: definition, why it matters, how it works, intuition, pitfalls, when-to-use guidance, practical demos, and a short exercise.


## Learning Objectives

By the end of this notebook, you will be able to:

- Design ingestion/cleaning for mixed document types
- Compare fixed, recursive, semantic, and structure-aware chunking
- Enrich chunks with metadata for filters, ACLs, and citations
- Measure chunk health before paying for embeddings


## Ingestion & Cleaning

**Definition.** **Ingestion** loads sources into normalized text+metadata. **Cleaning** removes boilerplate while preserving structure worth retrieving.

**Why it matters.** Embedders amplify nav bars, cookie banners, and OCR noise into false neighbors.

**How it works.** Per-source extractors → unicode normalize → strip boilerplate → keep headings/tables → attach provenance → emit Document objects.

**Intuition.** Keep the story, delete the ads, label who/when.

**Common pitfalls.**
- Stripping headings that enable structure-aware chunking
- Losing tables that hold the answer
- No document versioning → stale answers
- Merging unrelated PDF pages silently

**When to use.** Always—even 'clean' wikis need front-matter and link handling.

```mermaid
flowchart LR
  S[Sources] --> X[Extract]
  X --> N[Normalize]
  N --> B[Boilerplate filter]
  B --> P[Provenance]
  P --> D[Documents]
```


In [ ]:
# Demo 1 — HTML-ish cleaning
import re, html
raw = "<nav>Home</nav><h1>Refund Policy</h1><p>Within&nbsp;60&nbsp;days.</p><footer>c</footer>"

def clean(text):
    text = re.sub(r"(?is)<(nav|footer|script|style).*?>.*?</\1>", " ", text)
    text = re.sub(r"(?is)<h1[^>]*>(.*?)</h1>", r"\n# \1\n", text)
    text = re.sub(r"(?is)<p[^>]*>(.*?)</p>", r"\n\1\n", text)
    text = re.sub(r"(?is)<[^>]+>", " ", text)
    return re.sub(r"\s+", " ", html.unescape(text)).strip()

print(clean(raw))


In [ ]:
# Demo 2 — Document with provenance
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any

@dataclass
class Document:
    doc_id: str
    text: str
    source_uri: str
    modified_at: str
    acl: list[str] = field(default_factory=list)
    metadata: dict[str, Any] = field(default_factory=dict)

print(Document("pol-v3", "# Refunds\n60 days", "https://wiki/refund",
               datetime.now(timezone.utc).isoformat(), ["role:support"]))


In [ ]:
# Demo 3 — sanitize control chars / NBSP
def sanitize(t: str) -> str:
    t = t.replace("\x00", "").replace("\u00a0", " ")
    return t.encode("utf-8", "ignore").decode("utf-8").strip()
print(repr(sanitize("Refunds\x00 within\u00a060 days")))


### Try it yourself — Ingestion & Cleaning

1. Keep image alt text while stripping Markdown images.
2. Add a content checksum for change detection.
3. List three boilerplate patterns from your intranet.


## Chunking Strategies

**Definition.** **Chunking** splits documents into retrieval units. Common strategies: fixed windows, recursive separators, semantic splits, structure-aware cuts.

**Why it matters.** Size and boundaries dominate recall and noise in the context window.

**How it works.** Target token size + overlap; prefer heading→paragraph→sentence splits; store parent ids for later parent-child retrieval.

**Intuition.** A chunk should make sense alone—like one clear slide.

**Common pitfalls.**
- One size for contracts and chat logs
- Zero overlap on boundary-spanning answers
- Chunking before cleaning
- Splitting code/tables atomically needed as wholes

**When to use.** Start recursive/structure-aware; add semantic splits only if eval gains.

### Comparison

| Strategy | Pros | Cons |
|----------|------|------|
| Fixed | Simple | Blind to meaning |
| Recursive | Respects paragraphs | Needs separators |
| Semantic | Topic-coherent | Costly |
| Structure-aware | Great for manuals | Needs parsers |
| Parent-child | Precision + context | More complexity |


In [ ]:
# Demo 1 — fixed size + overlap
def chunk_fixed(text, size=20, overlap=5):
    words = text.split(); out=[]; i=0
    while i < len(words):
        out.append(" ".join(words[i:i+size]))
        i += max(1, size - overlap)
    return out
parts = chunk_fixed(" ".join(f"w{i}" for i in range(60)))
print(len(parts), parts[0].split()[-5:], parts[1].split()[:5])


In [ ]:
# Demo 2 — structure-aware-ish Markdown chunking
import re

def chunk_md(md, max_chars=160):
    sections = re.split(r"(?=\n#{1,3}\s)", "\n" + md.strip())
    out = []
    for sec in sections:
        sec = sec.strip()
        if not sec: continue
        if len(sec) <= max_chars:
            out.append(sec); continue
        buf = ""
        for para in re.split(r"\n\s*\n", sec):
            para = para.strip()
            if not para: continue
            if len(buf) + len(para) + 2 <= max_chars:
                buf = (buf + "\n\n" + para).strip()
            else:
                if buf: out.append(buf)
                buf = para
        if buf: out.append(buf)
    return out

sample = "# Shipping\n\nWe ship in 3-5 days.\n\n## Intl\n\nDuties may apply." + (" Extra." * 30)
for i, c in enumerate(chunk_md(sample)):
    print(i, len(c), c[:70].replace("\n", " / "))


In [ ]:
# Demo 3 — chunk health report
from statistics import mean

def health(chunks):
    lens = [len(c.split()) for c in chunks]
    return {"n": len(lens), "mean": round(mean(lens),1), "tiny": sum(x<20 for x in lens),
            "huge": sum(x>400 for x in lens)}
print(health(["x"]*3 + [" ".join(["w"]*80)]*2 + [" ".join(["w"]*500)]))


In [ ]:
# Demo 4 — sentence packing
import re

def chunk_sents(text, max_words=12):
    sents = re.split(r"(?<=[.!?])\s+", text.strip())
    out, buf, n = [], [], 0
    for s in sents:
        w = len(s.split())
        if buf and n + w > max_words:
            out.append(" ".join(buf)); buf, n = [s], w
        else:
            buf.append(s); n += w
    if buf: out.append(" ".join(buf))
    return out
print(chunk_sents("Refunds are easy. You have sixty days. Keep your receipt. Ask support."))


### Try it yourself — Chunking Strategies

1. Compare overlap vs no-overlap on a boundary-spanning question.
2. Pick different max sizes for FAQ vs API reference.
3. Flag near-duplicate adjacent chunks via Jaccard on word sets.


## Metadata Enrichment

**Definition.** **Metadata enrichment** attaches structured fields—source, section, time, ACL, product, language—so retrieval can filter, boost, and cite.

**Why it matters.** Dense similarity alone cannot enforce tenancy or recency preferences.

**How it works.** Propagate document fields to chunks; parse section paths; optionally extract entities; store payloads in the vector DB.

**Intuition.** Metadata is Dewey decimal; embeddings are blurbs—you need both.

**Common pitfalls.**
- Losing metadata at chunk boundaries
- Uncontrolled free-text tags
- PII in metadata logs
- Filters so tight recall collapses

**When to use.** Day one for multi-tenant/multi-product/regulated corpora.

| Field | Use |
|-------|-----|
| source_uri | Citations |
| section_path | Hierarchy |
| modified_at | Recency |
| tenant_id / acl | Security |
| doc_type | Routing/boost |


In [ ]:
# Demo 1 — propagate metadata to chunks
def enrich(doc_meta, section, text, i):
    return {"chunk_id": f"{doc_meta['doc_id']}::{i}", "text": text,
            "metadata": {**doc_meta, "section_path": section, "chunk_index": i}}
print(enrich({"doc_id": "pol-1", "tenant": "acme"}, "Policies/Refunds", "60 days", 0))


In [ ]:
# Demo 2 — ACL + product filter
rows = [
    {"id": "a", "product": "shop", "acl": ["support"]},
    {"id": "b", "product": "shop", "acl": ["legal"]},
    {"id": "c", "product": "saas", "acl": ["support"]},
]
def filt(rows, product=None, role=None):
    return [r for r in rows if (not product or r["product"]==product)
            and (not role or role in r["acl"])]
print(filt(rows, product="shop", role="support"))


In [ ]:
# Demo 3 — lightweight entity tags
import re
def tag(text):
    return {"days": list(map(int, re.findall(r"(\d+)\s*days", text.lower()))),
            "skus": re.findall(r"SKU[- ]?\d+", text, re.I)}
print(tag("Refund within 60 days for SKU-1234"))


### Try it yourself — Metadata Enrichment

1. Design a metadata schema for a multi-product help center.
2. Enforce tenant_id equality in a filter (no leakage).
3. Implement score' = score + α * freshness(modified_at).


## Glossary

- **provenance**: Origin and validity of a chunk
- **ACL**: Who may retrieve a chunk


### Workshop drill — Document Processing (1)

Diagram the data flow on paper, then implement one missing log line per stage.

Capture what you changed and what metric moved.


In [ ]:
# Workshop drill 1 — Document Processing
stages = ['ingest','chunk','embed','retrieve','pack','generate']
for s in stages:
    print(f'log.{{s}}.ok = ?')


### Workshop drill — Document Processing (2)

Create two adversarial queries (one ID-heavy, one paraphrase-heavy) and compare hits.

Capture what you changed and what metric moved.


In [ ]:
# Workshop drill 2 — Document Processing
queries = ['error code E42-refund', 'how do I get my money back?']
for q in queries:
    print('Q:', q)
    print('  TODO: print top-3 ids')


### Workshop drill — Document Processing (3)

Write a refusal test: empty hits must not produce a confident numeric answer.

Capture what you changed and what metric moved.


In [ ]:
# Workshop drill 3 — Document Processing
def must_refuse(hits):
    return (not hits) or hits[0].get('score',0) < 0.2
assert must_refuse([])
assert must_refuse([{'score': 0.05}])
assert not must_refuse([{'score': 0.9}])
print('refusal tests ok')


### Workshop drill — Document Processing (4)

Estimate cost: vary top_k and context tokens; print a small table.

Capture what you changed and what metric moved.


In [ ]:
# Workshop drill 4 — Document Processing
rows = []
for k in [2,4,8,16]:
    toks = k*400
    rows.append((k, toks, round(toks/1e6*0.5, 5)))
print('k  ctx_toks  approx_$')
for r in rows:
    print(*r)


### Workshop drill — Document Processing (5)

Add one metadata field and filter it in retrieval (tenant or product).

Capture what you changed and what metric moved.


In [ ]:
# Workshop drill 5 — Document Processing
docs = [{'id':'a','tenant':'acme'},{'id':'b','tenant':'beta'}]
tenant='acme'
print([d for d in docs if d['tenant']==tenant])


## Summary & Key Takeaways

- Cleaning beats clever retrievers on dirty corpora
- Chunk boundaries are a product decision—validate with real questions
- Metadata enables security, filters, and citations
- Measure chunk health before embedding at scale

### Practice

Ingest one PDF and one HTML page; compare chunk health reports.


## Self-Check

1. Can you explain the main idea of each section in one sentence?
2. Which technique would you use first in production, and why?
3. What failure mode should you monitor after shipping?
4. What metric would tell you the system got worse?


In [ ]:
checklist = [
    "I can restate the learning objectives",
    "I ran/adapted at least two code examples",
    "I know which env vars/keys this topic needs",
    "I noted one risk (cost, safety, latency, or quality)",
    "I can name one pitfall and its mitigation",
]
for i, item in enumerate(checklist, 1):
    print(f"{i}. [ ] {item}")
